# Pronóstico de Conteos Mensuales de Siniestros de Seguro de Auto con PROC FORECAST


## Resumen Ejecutivo

Un equipo actuarial de reservas necesita una vista a 12 meses de los conteos mensuales de siniestros de seguro de auto que muestre un crecimiento estable de la cartera más un marcado repunte por clima invernal. Este cuaderno genera cinco años de conteos mensuales sintéticos de siniestros (60 meses, ene 2020 - dic 2024, en un rango de aproximadamente 1.460 a 2.780 siniestros), y luego usa **PROC FORECAST** para ajustar una línea base autorregresiva escalonada y un modelo estacional multiplicativo de Holt-Winters. Cada modelo produce 12 pronósticos puntuales mensuales con límites de confianza del 95% para la planificación de capacidad y reservas. El modelo estacional Holt-Winters proyecta la demanda más fuerte de uno a dos meses adelante (aproximadamente 2.780-2.900 siniestros), suavizándose hacia un valle alrededor del paso nueve (aproximadamente 2.200), mientras que la línea base autorregresiva proyecta una disminución más suave; ambas bandas de confianza se ensanchan con el horizonte.


## Fuentes de Datos

| Conjunto de Datos | Filas | Granularidad | Variables Clave | Descripción |
|---------|------|-------|---------------|-------------|
| `claims` | 60 | Una fila por mes calendario (ene 2020 - dic 2024) | `date` (fecha SAS, `MONYY7.`), `claim_count` (numérica) | Conteos mensuales sintéticos de siniestros de seguro de auto construidos a partir de una tendencia de crecimiento lineal (~9 siniestros/mes), un ciclo anual sinusoidal, un repunte aditivo por clima invernal en dic/ene/feb, y ruido gaussiano (`rand('normal')`). La semilla `20240531` lo hace reproducible. |


# Pronóstico de Conteos Mensuales de Siniestros de Seguro de Auto

Los equipos de reservas y planificación de capacidad de una aseguradora de líneas personales necesitan una visión prospectiva de cuántos **siniestros de auto** se reportarán cada mes. El volumen de siniestros en este libro crece de manera constante a medida que la cartera se expande, y se dispara cada invierno cuando el hielo, la nieve y la reducción de luz diurna aumentan los siniestros de colisión y de cristales.

Este cuaderno recorre un flujo de trabajo completo de `PROC FORECAST`:

1. Generar una serie mensual de conteo de siniestros realista y completamente sintética.
2. Ajustar una línea base **autorregresiva escalonada (STEPAR)** que captura la tendencia más la autocorrelación.
3. Ajustar un modelo **multiplicativo de Holt-Winters (WINTERS)** que modela explícitamente el ciclo estacional de 12 meses.
4. Comparar los dos modelos e interpretar el pronóstico prospectivo y la banda de confianza.

Todo se ejecuta con datos sintéticos en línea — sin archivos externos ni acceso a la red.


## Paso 1 — Generar la serie sintética de siniestros

El paso DATA a continuación construye **60 observaciones mensuales** (ene 2020 a dic 2024). Para cada mes combinamos cuatro componentes que reflejan un libro de auto real:

- **Tendencia** — una línea base de ~1.800 siniestros que crece ~9 por mes a medida que aumenta la exposición.
- **Ciclo anual** — un término seno/coseno que da una onda estacional suave.
- **Repunte invernal** — un incremento aditivo en diciembre, enero y febrero.
- **Ruido** — `rand('normal', 0, 90)` para una variabilidad mes a mes realista.

`call streaminit()` fija el flujo aleatorio para que el cuaderno sea reproducible. La variable `date` es una fecha SAS verdadera construida con `INTNX` y formateada con `MONYY7.`, y la instrucción `ID date INTERVAL=MONTH` la designa como el identificador temporal. Las primeras 14 filas impresas a continuación caen aproximadamente entre 1.460 y 2.450 siniestros.


In [1]:
DATOS claims;
    LLAMAR streaminit(20240531);
    HACER i = 0 HASTA 59;
        /* Construir una fecha de mes SAS real para que INTERVAL=MONTH se alinee */
        date = intnx('month', '01JAN2020'd, i);
        FORMATO date monyy7.;

        month_idx = mod(i, 12) + 1;          /* 1 = ene ... 12 = dic */

        trend  = 1800 + 9 * i;               /* base de exposición creciente */
        season = 260 * sin((month_idx - 1) / 12 * 2 * constant('pi'))
               + 150 * cos((month_idx - 1) / 12 * 2 * constant('pi'));
        winter = 220 * (month_idx IN (12, 1, 2));   /* aumento por hielo/nieve */
        noise  = rand('normal', 0, 90);

        claim_count = round(trend + season + winter + noise);
        SI claim_count < 0 ENTONCES claim_count = 0;
        SALIDA;
    END;
    MANTENER date claim_count;
EJECUTAR;

PROCEDIMIENTO IMPRIMIR DATOS=claims(obs=14) ETIQUETA;
    ETIQUETA date = 'Mes' claim_count = 'Siniestros Reportados';
    TÍTULO 'Primeros 14 Meses de Conteos Sintéticos de Siniestros de Auto';
EJECUTAR;

                             Primeros 14 Meses de Conteos Sintéticos de Siniestros de Auto                              

  Obs    Mes  Siniestros Reportados
    1  21915                   2178
    2  21946                   2281
    3  21975                   2252
    4  22006                   1974
    5  22036                   2067
    6  22067                   1805
    7  22097                   1697
    8  22128                   1619
    9  22159                   1462
   10  22189                   1688
   11  22220                   1713
   12  22250                   2008
   13  22281                   2116
   14  22312                   2451

... 46 more observations (showing 14 of 60)




NOTE: DATA claims


NOTE: Wrote claims (60 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC PRINT data=claims

NOTE: PROC PRINT completed: 14 observations printed, 2 variables


## Paso 2 — Línea base autorregresiva escalonada (METHOD=STEPAR)

`METHOD=STEPAR` es el valor predeterminado. Primero ajusta un modelo de tendencia temporal — aquí `TREND=2` para una tendencia lineal — y luego aplica una **autorregresión escalonada** a los residuos, incorporando y reteniendo rezagos AR según su significancia. `AR=4` limita el orden autorregresivo candidato a cuatro rezagos, lo cual es suficiente para una serie mensual con impulso de corto plazo.

Opciones clave usadas aquí:

- `LEAD=12` — pronosticar 12 meses más allá de los datos.
- `ALPHA=0.05` — límites de confianza del 95%.
- `OUTFULL` — apilar las filas históricas `ACTUAL` junto con las filas del horizonte de pronóstico en el conjunto de datos `OUT=` (en lugar de solo las filas de pronóstico).
- `OUTLIMIT` — agregar las **columnas** de límite de confianza inferior/superior (`L95_claim_count`, `U95_claim_count`).
- `ID date INTERVAL=MONTH` — nombra el identificador temporal y declara que la serie es mensual.


In [2]:
PROCEDIMIENTO forecast DATOS=claims
        out=fc_stepar
        METHOD=stepar trend=2 ar=4
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    VAR claim_count;
EJECUTAR;

PROCEDIMIENTO IMPRIMIR DATOS=fc_stepar(obs=10) ETIQUETA;
    ETIQUETA date = 'Mes' claim_count = 'Siniestros' _type_ = 'Tipo'
          l95_claim_count = 'Límite Inf. 95%' u95_claim_count = 'Límite Sup. 95%';
    TÍTULO 'Salida STEPAR: Filas de Real, Ajustado y Pronóstico';
EJECUTAR;

                             Primeros 14 Meses de Conteos Sintéticos de Siniestros de Auto                              

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                  Salida STEPAR: Filas de Real, Ajustado y Pronóstico                                   

  Obs    Mes    Tipo   Siniestros   Límite Inf. 95%   Límite Sup. 95%
    1  21915  ACTUAL  2121.816667                 .                 .
    2  21946  ACTUAL  2167.711419                 .                 .
    3  21975  ACTUAL  2254.781177                 .                 .
    4  22006  ACTUAL  2228.505912                 .                 .
    5  22036  ACTUAL  1978.152296                 .                 .
    6  22067  ACTUAL  2030.606563                 .                 .
    7  22097  ACTUAL  1864.520418                 .                 .
    8  22128  ACTUAL  1784.855682                 .                 .
    9  22159  ACT


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: PROC PRINT data=fc_stepar

NOTE: PROC PRINT completed: 10 observations printed, 5 variables


### Leyendo el conjunto de datos OUT=

El conjunto de datos `OUT=` apila las filas mediante una columna `_TYPE_` y lleva los límites de confianza como **columnas** adicionales:

| Elemento | Tipo | Significado |
|---------|------|---------|
| `_TYPE_ = 'ACTUAL'` | fila | El `claim_count` observado para cada uno de los 60 meses históricos. |
| `_TYPE_ = 'FORECAST'` | fila | Las 12 predicciones puntuales para el horizonte de pronóstico. |
| `L95_claim_count` / `U95_claim_count` | columna | Límites de confianza inferior / superior del 95%, presentes en las filas `FORECAST` (ausentes en las filas `ACTUAL`). El nivel numérico refleja `ALPHA=`. |

Así, el `OUT=` impreso aquí contiene 72 filas: 60 filas históricas `ACTUAL` seguidas de 12 filas de horizonte `FORECAST`. Las primeras diez filas mostradas arriba son todas `ACTUAL`, con las columnas de límite de confianza ausentes porque los límites solo se adjuntan a las filas de pronóstico.

> **Nota del motor.** SAS `OUTFULL` también escribe una fila `FORECAST` de un paso adelante dentro de la muestra para cada mes histórico, y `OUTRESID` agrega filas `_TYPE_='RESIDUAL'`. PROC FORECAST de Jenner aún no emite esas filas ajustadas/residuales dentro de la muestra (registrado como prueba de brecha `400979`), por lo que este cuaderno lee solo el histórico `ACTUAL` y el horizonte prospectivo `FORECAST`.


## Paso 3 — Modelo estacional Holt-Winters (METHOD=WINTERS)

El modelo STEPAR captura la tendencia y la autocorrelación de corto plazo, pero no modela explícitamente el repunte recurrente de diciembre a febrero. Para una serie con un ritmo anual claro, **Holt-Winters multiplicativo** es la mejor herramienta.

`METHOD=WINTERS` descompone la serie en nivel, tendencia lineal (`TREND=2`), y un **factor estacional multiplicativo**. `SEASONS=12` declara un ciclo estacional de 12 periodos (mensual). Nuevamente solicitamos un `LEAD` de 12 meses con límites del 95% y `OUTFULL OUTLIMIT` para que la salida se alinee con la ejecución de STEPAR.

Como el motor avanza el `ID` de pronóstico en una unidad por paso (en lugar de respetar `INTERVAL=MONTH` para las fechas del horizonte — prueba de brecha `400979`), la celda siguiente revisa el pronóstico por **meses adelante (paso 1-12)** en lugar de basarse en las fechas de calendario impresas.


In [3]:
PROCEDIMIENTO forecast DATOS=claims
        out=fc_winters
        METHOD=winters seasons=12 trend=2
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    VAR claim_count;
EJECUTAR;

/* Mantener el horizonte prospectivo de 12 meses e indexarlo por paso (1..12). */
DATOS horizon;
    ESTABLECER fc_winters;
    RETENER months_ahead 0;
    SI _type_ = 'FORECAST';
    months_ahead + 1;
    MANTENER months_ahead claim_count l95_claim_count u95_claim_count;
EJECUTAR;

PROCEDIMIENTO IMPRIMIR DATOS=horizon ETIQUETA;
    ETIQUETA months_ahead     = 'Meses Adelante'
          claim_count       = 'Siniestros Pronosticados'
          l95_claim_count   = 'Límite Inf. 95%'
          u95_claim_count   = 'Límite Sup. 95%';
    TÍTULO 'Pronóstico Holt-Winters a 12 Meses (por paso)';
EJECUTAR;

                                  Salida STEPAR: Filas de Real, Ajustado y Pronóstico                                   

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                     Pronóstico Holt-Winters a 12 Meses (por paso)                                      

  Obs  Meses Adelante  Siniestros Pronosticados   Límite Inf. 95%   Límite Sup. 95%
    1               1                2783.07951       2577.844742       2988.314278
    2               2               2897.355557       2607.109764       3187.601349
    3               3               2805.272075       2449.795029        3160.74912
    4               4               2664.498049       2254.028514       3074.967585
    5               5               2628.810136       2169.891244       3087.729029
    6               6                2548.39319       2045.672732       3051.113649
    7               7               2391.649756         184


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: DATA horizon


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote horizon (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=horizon

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Paso 4 — Comparar los dos modelos cara a cara

La forma más clara de juzgar si el modelo estacional se justifica es colocar su pronóstico a 12 meses junto a la línea base STEPAR, paso a paso. Extraemos las filas `FORECAST` de ambos conjuntos de datos `OUT=`, indexamos cada una por meses-adelante, y las combinamos para que la divergencia sea visible de un vistazo.

Los dos métodos coinciden en el nivel general pero difieren en la **forma**: Holt-Winters lleva el ritmo anual al horizonte (un nivel más alto al inicio del horizonte que se suaviza hacia un valle a mitad del horizonte y vuelve a subir), mientras que STEPAR — que modela la estacionalidad solo indirectamente a través de rezagos AR — decae de forma más suave hacia la media de la serie. La brecha entre ambos en cada paso es la señal estacional que STEPAR deja sobre la mesa.

> SAS normalmente impulsaría esta verificación de adecuación con residuos de un paso adelante `OUTRESID` (`_TYPE_='RESIDUAL'`). Jenner aún no completa esas filas (prueba de brecha `400979`), por lo que en su lugar comparamos directamente los dos pronósticos prospectivos — un diagnóstico que usa solo la salida que el motor realmente produce.


In [4]:
/* Horizonte STEPAR, indexado por meses-adelante. */
DATOS stepar_h;
    ESTABLECER fc_stepar;
    RETENER months_ahead 0;
    SI _type_ = 'FORECAST';
    months_ahead + 1;
    stepar = claim_count;
    MANTENER months_ahead stepar;
EJECUTAR;

/* Horizonte WINTERS, indexado por meses-adelante. */
DATOS winters_h;
    ESTABLECER fc_winters;
    RETENER months_ahead 0;
    SI _type_ = 'FORECAST';
    months_ahead + 1;
    winters = claim_count;
    MANTENER months_ahead winters;
EJECUTAR;

/* Combinar ambos y mostrar la brecha estacional que STEPAR no captura. */
DATOS COMPARE;
    COMBINAR stepar_h winters_h;
    POR months_ahead;
    seasonal_gap = winters - stepar;
EJECUTAR;

PROCEDIMIENTO IMPRIMIR DATOS=COMPARE ETIQUETA;
    ETIQUETA months_ahead = 'Meses Adelante'
          stepar        = 'Pronóstico STEPAR'
          winters       = 'Pronóstico Winters'
          seasonal_gap  = 'Winters - STEPAR';
    FORMATO stepar winters seasonal_gap 8.0;
    TÍTULO 'STEPAR vs Holt-Winters: Comparación de Pronóstico a 12 Meses';
EJECUTAR;

                              STEPAR vs Holt-Winters: Comparación de Pronóstico a 12 Meses                              

  Obs  Meses Adelante   Pronóstico STEPAR   Pronóstico Winters  Winters - STEPAR
    1               1                2619                 2783               164
    2               2                2537                 2897               361
    3               3                2384                 2805               421
    4               4                2214                 2664               450
    5               5                2089                 2629               540
    6               6                2010                 2548               539
    7               7                1982                 2392               410
    8               8                1995                 2240               245
    9               9                2031                 2197               166
   10              10                2075                 2354      


NOTE: DATA stepar_h


NOTE: Read 72 rows from fc_stepar.
NOTE: Wrote stepar_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA winters_h


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote winters_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA compare

NOTE: Stream 1 processed 12 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 12 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote compare (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=compare

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Paso 5 — Pronosticar todas las líneas de negocio a la vez (procesamiento BY)

Las corridas de reservas reales cubren varios productos. Con los datos ordenados por `line_of_business`, una instrucción `BY` hace que `PROC FORECAST` ajuste un **modelo estacional independiente para cada grupo** en una sola llamada. Aquí sintetizamos un libro de Auto (mayor volumen base) y un libro de Hogar (base menor) y pronosticamos ambos a seis meses. `OUTALL` escribe el conjunto completo de filas — el histórico `ACTUAL` y el horizonte `FORECAST` — junto con las columnas de límite para cada grupo, y conservamos solo las filas `FORECAST` para la tabla siguiente.


In [5]:
DATOS multi_book;
    LLAMAR streaminit(20240531);
    LONGITUD line_of_business $6;
    HACER lob = 1 HASTA 2;
        SI lob = 1 ENTONCES line_of_business = 'Auto';
        SINO            line_of_business = 'Hogar';
        HACER i = 0 HASTA 47;
            date = intnx('month', '01JAN2021'd, i);
            FORMATO date monyy7.;
            mi   = mod(i, 12) + 1;
            BASE = (lob = 1) * 2000 + (lob = 2) * 1400;
            claim_count = round(BASE + 8 * i
                + 200 * sin((mi - 1) / 12 * 2 * constant('pi'))
                + 180 * (mi IN (12, 1, 2))
                + rand('normal', 0, 70));
            SALIDA;
        END;
    END;
    MANTENER line_of_business date claim_count;
EJECUTAR;

PROCEDIMIENTO ORDENAR DATOS=multi_book;
    POR line_of_business date;
EJECUTAR;

PROCEDIMIENTO forecast DATOS=multi_book
        out=fc_by
        METHOD=winters seasons=12 trend=2
        LEAD=6 ALPHA=0.05
        outall;
    POR line_of_business;
    id date interval=month;
    VAR claim_count;
EJECUTAR;

PROCEDIMIENTO IMPRIMIR DATOS=fc_by(obs=12) ETIQUETA;
    ETIQUETA line_of_business = 'Línea de Negocio'
          date             = 'Mes'
          claim_count      = 'Siniestros'
          _type_           = 'Tipo'
          l95_claim_count  = 'Límite Inf. 95%'
          u95_claim_count  = 'Límite Sup. 95%';
    DONDE _type_ = 'FORECAST';
    TÍTULO 'Pronósticos a 6 Meses por Línea de Negocio';
EJECUTAR;

                              STEPAR vs Holt-Winters: Comparación de Pronóstico a 12 Meses                              


BY Group: line_of_business=Auto

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6


BY Group: line_of_business=Hogar

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6



                                       Pronósticos a 6 Meses por Línea de Negocio                                       

  Obs   Línea de Negocio    Mes      Tipo   Siniestros   Límite Inf. 95%   Límite Sup. 95%  RESIDUAL_CLAIM_COUNT
    1  Auto               23742  FORECAST  2524.596717       2359.095846       2690.097589                     .
    2  Auto               23773  FORECAST  2604.418724       2370.365147         2838.4723                     .
    3  Auto               23801  FORECAST  2576.092313       2289.436395        2862.74823                     .
    4  Auto          


NOTE: DATA multi_book


NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC SORT data=multi_book

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 96 rows from multi_book.
NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC FORECAST data=multi_book

NOTE: BY Group: line_of_business=Auto
NOTE: Using Python for FORECAST estimation
NOTE: BY Group: line_of_business=Hogar
NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 108 observations.
NOTE: PROC PRINT data=fc_by

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 6 observations printed, 7 variables


## Interpretando los resultados

**Qué le dicen los modelos al equipo de reservas:**

- **STEPAR** reproduce la deriva ascendente y el impulso de corto plazo, pero su pronóstico a 12 meses decae suavemente desde aproximadamente 2.620 (paso 1) hacia unos 1.980 a mitad de horizonte antes de volver a subir a cerca de 2.140 — no reproduce un pico invernal marcado, porque trata la estacionalidad solo a través de rezagos autorregresivos. Es una línea base rápida razonable.
- **WINTERS** con `SEASONS=12` lleva el ritmo anual directamente a través de sus factores estacionales multiplicativos: su pronóstico es más alto al inicio del horizonte (aproximadamente 2.780 en el paso 1, aproximadamente 2.900 en el paso 2), se suaviza hacia un valle cerca del paso 9 (aproximadamente 2.200), y vuelve a subir hacia el paso 12 (aproximadamente 2.800). Frente a la línea base STEPAR se ubica **150-660 siniestros más alto** durante la mayor parte del horizonte (véase la columna `Winters - STEPAR` en el Paso 4) — esa brecha es la señal estacional que el modelo autorregresivo deja fuera.
- La **banda de confianza del 95%** (columnas `L95_*`/`U95_*`, controladas por `ALPHA=`) se ensancha a medida que se extiende el horizonte — para el modelo WINTERS, de un ancho de aproximadamente 410 siniestros en el paso 1 a cerca de 1.420 en el paso 12 — la señal honesta de que las estimaciones de horizonte lejano llevan más incertidumbre que las de corto plazo. Los actuarios de reservas deberían reservar capital contra el límite superior, no solo contra el pronóstico puntual.
- El **procesamiento BY** produce los pronósticos de Auto y Hogar en una sola pasada, cada uno con su propio ajuste estacional. El libro de Auto pronostica en el rango aproximado de 2.510-2.600 mientras que el libro de Hogar se ubica cerca de 1.870-2.030, de modo que cada línea conserva su propio nivel y forma estacional — el patrón que el equipo extendería a cada producto de la cartera.

**Conclusión:** para una serie de conteo de siniestros con un ciclo anual claro, `METHOD=WINTERS SEASONS=12` es la opción idiomática; la línea base STEPAR es una verificación de cordura útil, y `OUTFULL`/`OUTLIMIT` junto con una comparación de modelos paso a paso permiten al actuario dimensionar la reserva prospectiva con su banda de incertidumbre.

> **Nota de fidelidad del motor.** Este cuaderno documenta dos limitaciones actuales de PROC FORECAST en Jenner (prueba de brecha `400979`): el `ID` del horizonte de pronóstico se avanza en una unidad por paso en lugar de por `INTERVAL=MONTH`, por lo que las fechas de pronóstico impresas no son los meses de calendario de 2025 previstos (en su lugar revisamos el horizonte por paso); y `OUTRESID`/`OUTALL` aún no completan las filas `_TYPE_='RESIDUAL'` de un paso adelante, por lo que los diagnósticos residuales se reemplazan por una comparación directa de dos modelos. Los pronósticos puntuales y los límites de confianza en sí se producen y son en lo que se fundamenta la narrativa anterior.
